# **Lista interativa de estabelecimentos que oferecem serviços especializados para vítimas de violência sexual**

Esse notebook tem como objetivo criar um mecanismo de busca por estado e município de estabelecimentos que oferecem serviços especializados para vítimas de violência sexual. Pretende-se contribuir para que as vítimas tenham acesso a essa informação e consigam buscar atendimento de maneira mais célere e facilitada.

Para que o mecanismo de busca funcione, basta seguir os seguintes passos:

1. No topo da página, clicar no ícone "Executar tudo"
2. Na última célula, selecione a sigla do estado (UF) e o munícipio que deseja consultar
3. Pronto! Aparecerá em sua tela o nome do município, seguido nome do estabelecimento que oferece o serviço e do tipo de serviço ofertado.


In [ ]:
# @title
import pandas as pd

In [ ]:
# @title
df = pd.read_csv("https://raw.githubusercontent.com/otaviofabbro/busca_estabelecimentos_servicos_especializados/main/estabelecimentos_servicos_especializados.csv")

In [ ]:
# @title
import ipywidgets as widgets
from IPython.display import display, HTML

u_fs = sorted(df['UF'].dropna().unique().tolist())

uf_label = widgets.HTML(value='<span style="font-size:14px; font-weight:500;">Estado (UF)</span>')
municipio_label = widgets.HTML(value='<span style="font-size:14px; font-weight:500;">Município</span>')

uf_dropdown = widgets.Dropdown(
    options=['Selecione o estado'] + u_fs,
    description='',
    disabled=False,
    continuous_update=False,
    layout=widgets.Layout(width='320px')
)

municipio_dropdown = widgets.Dropdown(
    options=[],
    description='',
    disabled=True,
    continuous_update=False,
    layout=widgets.Layout(width='320px')
)

output_area = widgets.Output()

uf_box = widgets.VBox([uf_label, uf_dropdown])
municipio_box = widgets.VBox([municipio_label, municipio_dropdown])

STYLES = """
<style>
.result-card {
    font-family: sans-serif;
    background: #ffffff;
    border: 1px solid #e5e7eb;
    border-radius: 12px;
    padding: 20px 24px;
    margin-top: 12px;
    max-width: 680px;
}
.result-header {
    font-size: 15px;
    font-weight: 600;
    color: #111827;
    margin: 0 0 4px 0;
}
.result-subheader {
    font-size: 12px;
    color: #9ca3af;
    margin: 0 0 20px 0;
}
.establishment-list {
    list-style: none;
    margin: 0;
    padding: 0;
    display: flex;
    flex-direction: column;
    gap: 6px;
}
.establishment-card {
    background: #f9fafb;
    border: 1px solid #f3f4f6;
    border-radius: 8px;
    padding: 10px 14px;
}
.establishment-name {
    font-size: 14px;
    font-weight: 600;
    color: #111827;
    margin: 0 0 2px 0;
}
.establishment-service {
    font-size: 11px;
    color: #9ca3af;
    margin: 0;
}
.empty-msg {
    font-size: 12px;
    color: #d1d5db;
    font-style: italic;
}
.info-msg {
    font-family: sans-serif;
    font-size: 13px;
    color: #9ca3af;
    font-style: italic;
    margin-top: 12px;
}
</style>
"""

def on_uf_change(change):
    with output_area:
        output_area.clear_output()
        selected_uf = change.new
        if selected_uf and selected_uf != 'Selecione o estado':
            filtered_municipios = sorted(
                df[df['UF'] == selected_uf]['MUNICÍPIO'].dropna().unique().tolist()
            )
            municipio_dropdown.options = ['Selecione o Município'] + filtered_municipios
            municipio_dropdown.disabled = False
            municipio_dropdown.value = 'Selecione o Município'
        else:
            municipio_dropdown.options = []
            municipio_dropdown.disabled = True

def on_municipio_change(change):
    with output_area:
        output_area.clear_output()
        selected_municipio = change.new

        if not selected_municipio or selected_municipio == 'Selecione o Município':
            display(HTML(STYLES + '<p class="info-msg">Selecione um município para ver os serviços disponíveis.</p>'))
            return

        filtered_df = df[df['MUNICÍPIO'] == selected_municipio]

        if filtered_df.empty:
            display(HTML(STYLES + f'<p class="info-msg">Nenhum serviço encontrado para {selected_municipio}.</p>'))
            return

        uf = uf_dropdown.value
        total = len(filtered_df['NOME FANTASIA'].dropna().unique())

        estabelecimentos = (
            filtered_df[['NOME FANTASIA', 'SERVIÇO CLASSIFICAÇÃO']]
            .dropna(subset=['NOME FANTASIA'])
            .groupby('NOME FANTASIA')['SERVIÇO CLASSIFICAÇÃO']
            .apply(lambda x: sorted(x.dropna().unique().tolist()))
            .reset_index()
            .sort_values('NOME FANTASIA')
            .values.tolist()
        )

        html = STYLES
        html += '<div class="result-card">'
        html += f'<p class="result-header">{selected_municipio}</p>'
        html += f'<p class="result-subheader">{uf} &nbsp;·&nbsp; {total} estabelecimento{"s" if total != 1 else ""}</p>'

        html += '<ul class="establishment-list">'
        for name, services in estabelecimentos:
            services_html = ' &nbsp;·&nbsp; '.join(services)
            html += f'''<li class="establishment-card">
                <p class="establishment-name">{name}</p>
                <p class="establishment-service">{services_html}</p>
            </li>'''
        html += '</ul>'

        html += '</div>'
        display(HTML(html))

uf_dropdown.observe(on_uf_change, names='value')
municipio_dropdown.observe(on_municipio_change, names='value')

display(uf_box, municipio_box, output_area)

Output()